In [1]:
import pandas as pd

df = pd.read_parquet("yellow_tripdata_2024-01.parquet")
df.shape

(2964624, 19)

In [2]:
df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df["duration"] = df.duration.dt.total_seconds() / 60

df = df[(df.duration >= 1) & (df.duration <= 60) & (df.passenger_count > 0)]
df = df[(df.tpep_pickup_datetime >= "2024-01-01") & (df.tpep_pickup_datetime < "2024-02-01")]

df.shape

(2729769, 20)

In [4]:
df["pickup_date"] = df.tpep_pickup_datetime.dt.date

daily = df.groupby("pickup_date").agg(
    trips=("VendorID", "size"),
    avg_fare=("fare_amount", "mean"),
    avg_duration=("duration", "mean"),
    total_amount=("total_amount", "sum"),

)

daily.head(10)

,trips,avg_fare,avg_duration,total_amount
pickup_date,,,,
2024-01-01,67964,20.941670,14.820694,2033134.55
2024-01-02,70444,20.055820,15.249419,2067241.50
2024-01-03,77059,18.534897,14.722518,2110191.10
2024-01-04,97052,17.738459,14.776159,2576767.93
2024-01-05,96967,17.189877,14.246002,2505142.69
2024-01-06,89462,16.870632,13.341344,2216414.01
2024-01-07,63084,19.120532,13.021863,1760272.89
2024-01-08,75014,17.862576,14.106770,2015005.56
2024-01-09,83701,16.117000,13.978393,2056580.05


In [5]:
daily.to_parquet("taxi_daily_summary.parquet")